# Part 1 : Local VLMs & Network Inference

Welcome to the first module! In robotics, the "brain" (AI model) often dictates how responsive and capable the robot feels. Today, you will explore the trade-offs between running a highly quantized vision-language model (VLM) directly on edge hardware (your laptop) versus offloading heavy reasoning to a remote server.

**Goals for this section:**
1. Understand the memory footprint of local VLMs.
2. Benchmark Time-To-First-Token (TTFT) and Tokens-Per-Second (TPS).
3. Experience why **streaming** is mandatory for human-robot interaction.

In [ ]:
# Run this to avoid having a lot of logs from llama cpp C++ backed (NOT RECOMMENDED IN ACTUAL USE)
import os
import sys

# Open a black hole
devnull = os.open(os.devnull, os.O_WRONLY)

# 2. Globally redirect the OS-level stderr (C++ logs) into the black hole
os.dup2(devnull, 2)

print("C++ backend logs permanently silenced.")
print("Python error reporting remains fully active.")

### Loading the Local VLM

Before we can start chatting with our robot, we need to load our Vision-Language Model (VLM) . We are using **llama.cpp**, a highly optimized C++ engine that allows massive AI models to run smoothly on local (and not only) hardware.

To make this work, we need two specific files:
* **The Text Model (`model_path`):** This is the "brain" of the AI. It handles the logic, reasoning, and language generation.
* **The Vision Projector (`clip_model_path`):** These are the "eyes." It acts as a translator, converting raw image pixels into mathematical embeddings that the text model can understand. Basically, it is the vision backbone + projector that allows the text model to "see" and reason about images.

**What is a GGUF file?**
You will notice our files end in `.gguf`. This is a specialized file format designed for local AI. It bundles the model's architecture and quantized (compressed) weights into a single file, drastically reducing the RAM required to run it without significantly degrading its performance.

**The Magic of `n_gpu_layers` (VRAM Management)**
In the code below, look for the `n_gpu_layers` parameter.
* Setting it to `-1` tells the engine to load the *entire* model into your graphics card (GPU) for maximum speed.
* However, if your GPU doesn't have enough Video RAM (VRAM) to hold the whole model, it will crash! (If you use gemma you will need at least 8GB of VRAM to load the model fully. (4 for the model and 4 for the context window))
* The solution? You can set this to a specific number (e.g., `n_gpu_layers=10`). The engine will load exactly 10 layers into your fast GPU and leave the rest on your standard computer RAM (CPU). This hybrid approach will slow down token generation, but it gives you the superpower to run massive models that would otherwise be impossible on your hardware!
* You don't have a GPU? No worries! The engine will automatically detect this and run the model entirely on your CPU. It will be slower, but it will still work.

In [ ]:
import gc
import time
import os
import sys
from contextlib import contextmanager
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Llava15ChatHandler


# --- Choose Your Model (wisely or not) ---
# Option A: Gemma 4-Bit (Faster, uses less VRAM)
MODEL_PATH = "/app/models/gemma-4-E4B-it-Q4_K_M.gguf"
CLIP_PATH = "/app/models/mmproj-BF16.gguf"

# Option B:  FIXME: add one more model
# MODEL_PATH = "/app/models/your_other_model.gguf"
# CLIP_PATH = "/app/models/your_other_projector.gguf"

print(f"📥 Loading local VLM into memory...")
start = time.time()

# Initialize the "Eyes" (Vision Projector)
chat_handler = Llava15ChatHandler(
    clip_model_path=CLIP_PATH
)

# Initialize the "Brain" (Text Model)
local_vlm = Llama(
    model_path=MODEL_PATH,
    chat_handler=chat_handler,
    n_gpu_layers=-1, # Offloads all processing to the GPU
    n_ctx=8192,      # The maximum context window size
    verbose=False,
)

print(f"✅ Local VLM Ready! Load time: {time.time() - start:.2f}s")
# Silent Warm-up (Forces the C++ engine to allocate memory now, not later)
local_vlm.create_chat_completion(
    messages=[{"role": "user", "content": [{"type": "text", "text": "warmup"}]}],
    max_tokens=1
)


#### 🕵️‍♂️ Verifying the GPU Payload (GPU Only)

When you ran the cell above, you forced a multi-gigabyte AI model directly into your graphics card's Video RAM (VRAM). Let's prove it!

Open a new terminal window in your workspace and run one of the following commands:

* **`nvidia-smi`**: The standard NVIDIA tool. It takes a static snapshot of your current GPU state.
* **`nvtop`**: A visually awesome, real-time task manager specifically for GPUs (highly recommended!). (If you don't have it installed, you can install it with `sudo apt install nvtop`.)

Look at the **Memory Usage** section. You should see a large block of VRAM allocated to your Python process.

**Experiment:** If you are using the Gemma 4-Bit model, it should take up roughly 4-5 GB of VRAM. If you switch to the LLaVA 1.5 7B model in the code above and run it again, watch how much the memory footprint jumps! (Note, you will have to restart the kernel to load a new model)

### Your First(?) Local Multimodal Inference using llama.cpp

Now that the VLM is loaded, let's ask it a question about a local image! 

We will use the standard **OpenAI Chat formatting**. Instead of just sending a raw text string, we send a `messages` list. Because this is a *multimodal* model, the user's message isn't just text, it is a list containing both an image block and a text block.

**Important Note on Images:** These models cannot read local file paths like `C:/images/photo.jpg`. You must convert the image into a "Base64 Data URI" (a massive string of text that represents the raw image pixels) before handing it to the model. 

Let's test this out using an image from your `lab/resources/` folder.

In [ ]:
import base64

def image_to_base64_data_uri(file_path: str):
    with open(file_path, "rb") as img_file:
        base64_data = base64.b64encode(img_file.read()).decode('utf-8')
        return f"data:image/png;base64,{base64_data}"

# Point to a local image (Update this filename to try different images
test_image_path = "./resources/franka.png"

# Convert the image to a Base64 Data URI using our helper function
local_image_uri = image_to_base64_data_uri(test_image_path)

print(f"👀 Looking at {test_image_path}...")

# 3. Construct the prompt and run inference
response = local_vlm.create_chat_completion(
    messages=[{
        "role": "user",
        "content": [
            # 🖼️ The Image Block
            {"type": "image_url", "image_url": {"url": local_image_uri}},
            # 💬 The Text Block
            {"type": "text", "text": "Describe exactly what you see in this image."}
        ]
    }],
    max_tokens=250,     # Limit the response length for this quick test
    temperature=0.3     # Lower temperature = more factual, less creative, feel free to play around
)

# Extract and clean the text (removing any Llava template leakage)
raw_text = response['choices'][0]['message']['content'].strip()
if "ASSISTANT:" in raw_text:
    clean_text = raw_text.split("ASSISTANT:")[-1].strip()
else:
    clean_text = raw_text

print("\n🤖 VLM Response:")
print(clean_text)

## Activity 1: Measuring Latency (TTFT & TPS)

For a robot to feel alive and responsive, we care deeply about two metrics:
* **Time-To-First-Token (TTFT):** How long does the human have to wait before the robot starts reacting? Processing vision adds a heavy penalty here.
* **Tokens-Per-Second (TPS):** Once the robot starts talking, how fast does it generate words?

**Task 1:** Complete the code below to calculate the Generation Speed (TPS) for three different prompts. Notice how the length of your prompt impacts the speed!

In [ ]:
import time

prompts = [
    "What is this?",
    "Describe what is happening in front of you in detail.",
    "Is there a human visible in this image? Answer with exactly 'yes' or 'no.'"
]

# Point to a local image (Update this filename to try different images
test_image_path = "./resources/franka.png"

# Convert the image to a Base64 Data URI using our helper function
local_image_uri = image_to_base64_data_uri(test_image_path)


for idx, prompt_text in enumerate(prompts):
    print(f"\nEvaluating Prompt {idx + 1}: {prompt_text}")

    start_time = time.time()

    response = local_vlm.create_chat_completion(
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": local_image_uri}},
                {"type": "text", "text": prompt_text}
            ]
        }],
        max_tokens=150,
        temperature=0.7
    )

    end_time = time.time()
    total_latency = end_time - start_time
    tokens_generated = response['usage']['completion_tokens']

    print(response['choices'][0]['message']['content'])

    #TODO: calculate and replace the 0.0 placeholder below
    tokens_per_second = 0.0
    print(f"[Total time: {total_latency:.2f}s, Tokens generated: {tokens_generated}, Tokens per second: {tokens_per_second:.2f}]")

## Activity 2: The Magic of Streaming
In the previous cell, we used **blocking** inference. The Python script completely froze until the model finished generating every single word.

If a robot waits 4 seconds to think of a 20-word sentence, the human user will assume the robot is broken. Instead, we use **streaming**. By passing `stream=True`, we can grab words as they are generated in real-time and immediately send them to the Text-to-Speech (TTS) engine.

Think of any LLM you have used through a web interface. You probably noticed that the text appears word by word, not all at once. You can already start reading the response while the model is still generating the rest, which increases the perceived responsiveness of the system. This is exactly what we want for our robot!

In [ ]:
import time

prompt = "Who are you?"

# --- Test 1: Blocking ---
print("Test 1: Blocking (Waiting for full response...)")
start_blocking = time.time()
blocking_res = local_vlm.create_chat_completion(
    messages=[{"role": "user", "content": prompt}],
    max_tokens=60,
    temperature=0.4
)
total_blocking = time.time() - start_blocking
print(f"Result: {blocking_res['choices'][0]['message']['content'].strip()}")
print(f"Total latency: {total_blocking:.2f}s\n")

# --- Test 2: Streaming ---
print("Test 2: Streaming (Receiving words as they appear...)")
start_streaming = time.time()
stream = local_vlm.create_chat_completion(
    messages=[{"role": "user", "content": prompt}],
    max_tokens=60,
    temperature=0.4,
    stream=True
)

first_token_time = None
full_response = ""

print("Robot says: ", end="")

for chunk in stream:
    if first_token_time is None:
        first_token_time = time.time()
        ttft = first_token_time - start_streaming
        print(f"\n[⚡ TTFT: {ttft:.2f}s]", end="")

    delta = chunk["choices"][0]["delta"]
    if "content" in delta:
        content = delta["content"]
        full_response += content
        print(content, end="", flush=True)

total_streaming = time.time() - start_streaming
print(f"\n\n[🏁 Total Streaming Time: {total_streaming:.2f}s]")
print(f"[⏱️  Average Generation Speed: {(len(full_response.split())/total_streaming):.2f} words/sec]")

---
**End of Part 1.** You now know how to run vision models locally, how to stream tokens, and how to offload to a server. Head over to `part_2.ipynb` to learn how we force these models to output safe, robotic commands instead of conversational text! 

# Part 2: Prompt Engineering in Robotics

Moving a physical robot safely requires strict command structures. If your python script expects a dictionary with a motion command, but the LLM response is something like *"Sure, I'd love to help you wave!*, we will not be able to parse this command to move the robot, even with sophisticated regex.

In this module we will explore how to use prompt engineering to enable some prerecorded motions alongside natural language responses.

**Goals for this section:**
1. **System Personas:** Change the robot's fundamental behavior.
2. **JSON Schema Mode:** Force the model to return data structures, not string.
3. **Chain of Thought: (CoT)** Enable the model to reason about its actions before responding.
4. **Few shot prompting:** Give the model examples of how to respond to specific commands.
5. **GBNF Grammars:** Mathematically lock the model's output to a specific set of commands.

### Activity 1: Defining System Personas

In robotics, a **System Prompt** acts as the "foundational personality" of your agent. Unlike the `user_input` (which is the command given by a human), the `system_instruction` is the hidden configuration that defines the robot's identity, constraints, and operational logic. 

Because we are working with **Reachy Mini**, a social robot that is white, has two antennas, and a moving head, our persona should reflect its physical reality. Without a strong system prompt, an LLM will often "hallucinate" capabilities the robot doesn't have, like trying to pick up objects it cannot reach or walk when it has wheels.


By defining a system prompt, we provide the **frame of reference** for every interaction, ensuring the robot acts consistently whether it is feeling "happy" or "shy."

In [ ]:
def test_persona(system_instruction, user_input):
    res = local_vlm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_input}
        ],
        max_tokens=140,
        temperature=0.6
    )
    return res["choices"][0]["message"]["content"]


# --- Persona Definition ---

# A robust system prompt defining the robot's physical identity
persona_happy = (
    "You are Reachy Mini, a social robot with a white body, two antennas, and a moving head. "
    "You are always positive and incredibly happy. "
    "Keep your response to one short sentence. Do not use emojis."
)

# Placeholders for students to fill in
persona_shy = "TODO: Write a system prompt for a shy Reachy Mini."
persona_sad = "TODO: Write a system prompt for a depressed Reachy Mini."
persona_snob = "TODO: Write a system prompt for a pretentious, snobby Reachy Mini."

user_trigger = "The user is waving at you."

# Testing the personas
print(f"😊 Happy: {test_persona(persona_happy, user_trigger)}")
print(f"🥺 Shy:   {test_persona(persona_shy, user_trigger)}")
print(f"😞 Sad:   {test_persona(persona_sad, user_trigger)}")
print(f"🧐 Snob:  {test_persona(persona_snob, user_trigger)}")

### Activity 2: JSON Schema Mode

Robots need structured data. `llama-cpp-python` supports **JSON Schema Mode**, which forces the model to output a perfectly formatted JSON string that Python can immediately parse into a dictionary using `json.loads()`.

**Your Task:** The schema below is incomplete. Read the user input, and add the missing properties to the `robot_schema` so the model extracts the user's emotion and how urgent their request is.

In [ ]:
import json

# --- PART 1: The Schema Definition ---
# TODO: Add two missing properties to this schema:
# 1. "emotional_state" (Must be a string enum: ["happy", "sad", "frustrated", "neutral"])
# 2. "urgency_level" (Must be an integer)
#
# 💡 HINT - Here is how you define an enum in JSON Schema:
# "some_setting": {
#     "type": "string",
#     "enum": ["option_A", "option_B"]
# }

robot_schema = {
    "type": "object",
    "properties": {
        "user_intent": {
            "type": "string",
            "description": "A short description of what the user wants and how they feel."
        },
    },
    "required": ["user_intent", "emotional_state", "urgency_level"]
}

user_input = "My code has been compiling for 40 minutes and it just threw a syntax error. I am going to throw my laptop out the window right now."

# --- PART 2: Without Schema (The Wild West) ---
print("🚨 Test 1: Asking for JSON WITHOUT a strict schema...")
wild_response = local_vlm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Analyze the user statement and output clean JSON with user_intent, emotional_state, and urgency_level."},
        {"role": "user", "content": user_input},
    ],
    temperature=0.7
)
print(wild_response["choices"][0]["message"]["content"].strip())
print("-" * 50)

# --- PART 3: With Schema (Locked Down) ---
print("\n🔒 Test 2: Forcing JSON WITH our strict schema...")
json_response = local_vlm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Analyze the user statement and output clean JSON matching the schema."},
        {"role": "user", "content": user_input},
    ],
    # This flag forces the C++ backend to strictly follow your schema
    response_format={"type": "json_object", "schema": robot_schema},
    temperature=0.7
)

parsed_data = json.loads(json_response["choices"][0]["message"]["content"])

print("\n--- Safely Ingested Data Structure ---")
print(f"Intent: {parsed_data.get('user_intent', 'MISSING')}")
print(f"Emotion: {parsed_data.get('emotional_state', 'MISSING')}")
print(f"Urgency: {parsed_data.get('urgency_level', 'MISSING')}")

### Activity 3: Chain of Thought (Thinking Before Acting)

Small local models (and even giant cloud models) struggle with nuance, especially **sarcasm**. 

If a user says, *"Oh great, another robot. Just what I needed today. Fantastic."*, a naive model will see words like "great" and "fantastic" and immediately trigger a happy animation. 

**The Fix:** We use a technique called **Chain of Thought (CoT)**. By forcing the model to write out its reasoning *before* it makes a final decision, we give it space to "think." 

**The Golden Rule of JSON CoT:**
Models generate text linearly (top-to-bottom). Therefore, your `internal_monologue` key **must** appear in the schema *before* the `action` key. If the model picks the action first, the monologue is useless!

**Your Task:** 1. Add the `internal_monologue` to the schema. 
2. Write a system prompt commanding the model to detect sarcasm before selecting an action.

In [ ]:
import json

tricky_input = "Oh great, another robot. Just what I needed today. Fantastic."

# --- PART 1: The CoT Schema ---
# TODO: Add "internal_monologue" as a string type.
# Give it a description like: "Analyze the user's tone for sarcasm before picking an action."

cot_schema = {
    "type": "object",
    "properties": {
        # TODO: Add "internal_monologue" here!
        "action": {
            "type": "string",
            "enum": ["HAPPY_DANCE", "SAD_SIGH", "CONFUSED_HEAD_TILT", "IDLE"],
            "description": "The final physical action the robot should take."
        }
    },
    "required": ["internal_monologue", "action"]
}

# --- PART 2: The CoT Prompt ---
# TODO: Write a system prompt that explicitly tells the model to use its internal monologue
# to detect sarcasm and passive-aggressive tones before choosing an action.
system_prompt = """
"""

print(f"🗣️ User said: '{tricky_input}'")
print("🤖 Thinking...")

# --- PART 3: Inference ---
cot_response = local_vlm.create_chat_completion(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": tricky_input}
    ],
    response_format={"type": "json_object", "schema": cot_schema},
    temperature=0.1
)

cot_data = json.loads(cot_response["choices"][0]["message"]["content"])

print("\n--- Robot Decision Logic ---")
print(f"🧠 Internal Monologue: {cot_data.get('internal_monologue', 'MISSING')}")
print(f"🦾 Final Action: {cot_data.get('action', 'MISSING')}")

### Activity 4: GBNF Macro Locking

JSON Schema is great, but we may sometimes need a more complex or more simple grammar than JSON. For example, we may want to restrict the model to only output a specific set of macros that are safe for the robot to execute. This is where **GBNF (GGML BNF)** comes in.

**GBNF (GGML BNF)** allows us to restrict the model's token generation at the C++ engine level. It becomes physically impossible for the model to generate a sequence of characters outside your defined grammar.

**Your Task:** Reachy Mini expects one of four exact macros: `"HAPPY_DANCE"`, `"SAD_SIGH"`, `"CONFUSED_HEAD_TILT"`, or `"IDLE"`. The grammar below only knows one. Add the rest!

In [ ]:
from llama_cpp import LlamaGrammar

# --- PART 1: Define the Grammar ---
# The 'motion' rule currently only allows "HAPPY_DANCE".
# TODO: Use the OR operator `|` to add "\"SAD_SIGH\"", "\"CONFUSED_HEAD_TILT\"", and "\"IDLE\"".
# HINT: Look at how "HAPPY_DANCE" is formatted with escaped quotes.

# --- PART 1: Define the Grammar ---

gbnf_grammar = r"""
root   ::= ws "[" ws motion ws "]" ws
motion ::= "\"HAPPY_DANCE\""
ws     ::= [ \t\n]*
"""

print("📐 Compiling strict C++ grammar...")
compiled_grammar = LlamaGrammar.from_string(gbnf_grammar)

# --- PART 2: Test the Constraints ---
scenarios = [
    "The user just clapped and cheered for you!",
    "The user asks you a confusing question you do not understand about quantum physics."
]

for scenario in scenarios:
    response = local_vlm.create_chat_completion(
        messages=[
            {"role": "system", "content": "Select the single best macro motion representing the emotional context. Your options are HAPPY_DANCE, SAD_SIGH, CONFUSED_HEAD_TILT, IDLE"},
            {"role": "user", "content": scenario}
        ],
        grammar=compiled_grammar, # <-- Applying the ultimate lockdown
        max_tokens=20,
        temperature=0.
    )
    print(f"\nContext: '{scenario}'\n🎯 Locked Output: {response['choices'][0]['message']['content']}")

### Ativity 5: Few-Shot Prompting (Teaching by Example)

Our GBNF Grammar guarantees the robot won't crash by outputting invalid text, but as we just saw, the AI might still pick the *wrong* valid option (like choosing `IDLE` when it should be `CONFUSED_HEAD_TILT`).

Small AI models often fail at **Zero-Shot** tasks (guessing the right answer with no examples). 

**The Fix:** We use **Few-Shot Prompting**. We give the model a "fake" conversation history showing exactly how we expect it to behave before we ask it our real question. This acts as a logical blueprint.

In [ ]:
from llama_cpp import LlamaGrammar

gbnf_grammar = r"""
root   ::= ws "[" ws motion ws "]" ws
motion ::= "\"HAPPY_DANCE\"" | "\"SAD_SIGH\"" | "\"CONFUSED_HEAD_TILT\"" | "\"IDLE\""
ws     ::= [ \t\n]*
"""

compiled_grammar = LlamaGrammar.from_string(gbnf_grammar)

# We fixed the missing period at the end of the system prompt!
system_instruction = "Select the single best macro motion representing the emotional context. Your options are HAPPY_DANCE, SAD_SIGH, CONFUSED_HEAD_TILT, IDLE."

scenarios = [
    "The user asks you a confusing question you do not understand about quantum physics."
]

for scenario in scenarios:
    response = local_vlm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_instruction},

            # --- FEW-SHOT EXAMPLES (The Blueprint) ---
            {"role": "user", "content": "Can you explain string theory in Mandarin?"},
            {"role": "assistant", "content": '["CONFUSED_HEAD_TILT"]'},
            {"role": "user", "content": "Hey, nothing is happening right now."},
            {"role": "assistant", "content": '["IDLE"]'},
            # -----------------------------------------

            # The actual question!
            {"role": "user", "content": scenario}
        ],
        grammar=compiled_grammar,
        max_tokens=20,
        temperature=0.0
    )
    print(f"\nContext: '{scenario}'\n🎯 Locked Output: {response['choices'][0]['message']['content'].strip()}")

### Activity 6: GBNF JSON Grammars (The Boss Battle)

Even with JSON Schema, models can sometimes hallucinate or break formatting if the user's prompt is chaotic. What if you want 100% mathematical certainty that the model will ONLY output a perfectly structured JSON string, and literally cannot generate anything else?

Enter **GBNF (GGML BNF) Grammars**. 

Instead of *asking* the model to format its output, GBNF hooks directly into the C++ token generation engine at the lowest level. If the model tries to predict a character that breaks your JSON structure, the engine physically blocks that token from being generated. It is the most powerful safety tool for robotics.

**Your Task:** Let's combine everything we've learned! The grammar below is missing its core rules. 
1. Construct the `root` rule to build a JSON object with exactly three keys: `chain_of_thought`, `emotion`, and `response`.
2. Define the `emotion` rule to strictly allow only `"happy"`, `"sad"`, or `"neutral"`.

In [ ]:
import json
from llama_cpp import LlamaGrammar

#TODO: Write a GBNF grammar that forces the model to output a JSON object with three properties:
# 1. "chain_of_thought" (string)
# 2. "emotion" (string enum: ["happy", "sad", "neutral", "frustrated"])
# 3. "response" (string)
gbnf_json_grammar = r"""
# TODO 1: Write the root rule here.
root    ::=

# TODO 2: Define the allowed emotions.
emotion ::=

# --- PROVIDED PRIMITIVES ---
string  ::= "\"" [^"]+ "\""
ws      ::= [ \t\n]*
"""

print("Compiling GBNF JSON grammar...")
compiled_json_grammar = LlamaGrammar.from_string(gbnf_json_grammar)

scenario = "I tried to fix the robot's code, but I accidentally deleted the entire main file. I'm so sorry."

response = local_vlm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are Reachy Mini. Analyze the user's text, explain your reasoning, pick an emotion, and generate a short spoken response. OUTPUT EXACTLY AND ONLY RAW JSON. Do not output any conversational text or introductions."},
        {"role": "user", "content": scenario}
    ],
    grammar=compiled_json_grammar,
    temperature=0.6
)

raw_string = response['choices'][0]['message']['content'].strip()
print(f"\nGuaranteed Raw String:\n{raw_string}")

# Prove that Python can load it perfectly as a dictionary
try:
    parsed_dict = json.loads(raw_string)
    print("\n✅ Successfully Parsed into Python Dictionary!")
    print(f"Reasoning: {parsed_dict['chain_of_thought']}")
    print(f"Emotion Triggered: {parsed_dict['emotion']}")
    print(f"Robot Says: {parsed_dict['response']}")
except Exception as e:
    print(f"❌ Failed to parse JSON: {e}")

---
**End of Part 2.** You now know how to build a robust cognitive pipeline using Personas, JSON Schemas, Chain of Thought, and strict Grammars. 